# House Price Prediction — Clean Notebook
This notebook contains a compact, reproducible pipeline: imports, safe preprocessing, feature engineering, and a baseline RandomForest model. Run top-to-bottom.

In [37]:
# Imports
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

In [38]:
# Load datasets (adjust paths if needed)
train_df = pd.read_csv('Datasets/train_(2)_(1)_(1).csv')
test_df = pd.read_csv('Datasets/test_(2)_(1)_(1).csv')
avg_rent = pd.read_csv('Datasets/avg_rent_(1)_(1)_(1).csv')
print('train shape:', train_df.shape)
print('test shape :', test_df.shape)

train shape: (10656, 10)
test shape : (2664, 9)


In [39]:
# Safe preprocessing helpers
def get_size_value(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if 'rk' in s:
        return 0
    m = re.match(r'^(\d+)', s)
    if m:
        return int(m.group(1))
    return np.nan

def get_sq_ft(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    # direct float
    try:
        return float(s)
    except Exception:
        pass
    # range like '1200 - 1500'
    if '-' in s:
        parts = re.split(r'[-–]', s)
        nums = []
        for p in parts:
            num = ''.join(ch for ch in p if (ch.isdigit() or ch == '.'))
            if num:
                try:
                    nums.append(float(num))
                except:
                    pass
        if len(nums) == 2:
            return (nums[0] + nums[1]) / 2.0
    # fallback: first numeric token
    m = re.search(r'([\d\.]+)', s)
    if m:
        try:
            return float(m.group(1))
        except:
            return np.nan
    return np.nan

def loc_clean(x):
    if pd.isna(x):
        return x
    s = str(x).lower()
    s = re.sub(r'[,:!.\"\/\-()]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

In [40]:
# Minimal reproducible preprocessing (keeps it simple and robust)
df = train_df.copy()
# drop useless columns if present
for c in ['ID','society','availability']:
    if c in df.columns:
        df.drop(columns=c, inplace=True)

# clean and convert columns
if 'size' in df.columns:
    df['size'] = df['size'].apply(get_size_value)
if 'total_sqft' in df.columns:
    df['total_sqft'] = df['total_sqft'].apply(get_sq_ft)
if 'location' in df.columns:
    df['location'] = df['location'].apply(loc_clean)

# basic feature engineering
# if 'total_sqft' in df.columns and 'price' in df.columns:
#     df['price_per_sqft'] = df.apply(lambda r: (r['price'] / r['total_sqft']) if pd.notna(r['total_sqft']) and r['total_sqft']>0 else np.nan, axis=1)

# drop rows with missing target or essential numeric fields
df = df.dropna(subset=['price'])
print('After cleaning, rows:', df.shape[0])
df.head()

After cleaning, rows: 10656


,area_type,location,size,total_sqft,bath,balcony,price
0,Super built-up Area,electronic city phase ii,2.0,1056.0,2.0,1.0,39.07
1,Plot Area,chikka tirupathi,4.0,2600.0,5.0,3.0,120.00
2,Built-up Area,uttarahalli,3.0,1440.0,2.0,3.0,62.00
3,Super built-up Area,lingadheeranahalli,3.0,1521.0,3.0,1.0,95.00
4,Super built-up Area,kothanur,2.0,1200.0,2.0,1.0,51.00


In [41]:
# Prepare features and baseline RandomForest model
features = [f for f in ['size','total_sqft','bath','balcony','area_type','location',] if f in df.columns]
X = df[features].copy()
y = np.log1p(df['price'])
# simple train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# transformers
numeric_features = [c for c in ['size','total_sqft','bath','balcony',] if c in X.columns]
cat_features = [c for c in ['area_type','location'] if c in X.columns]
numeric_transformer = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_transformer = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor = ColumnTransformer([('num', numeric_transformer, numeric_features), ('cat', cat_transformer, cat_features)])
pipe = Pipeline([('pre', preprocessor), ('rf', LinearRegression())])
pipe.fit(X_train, y_train)
pred_log = pipe.predict(X_test)
rmse_log = mean_squared_error(y_test, pred_log)
rmse_price = mean_squared_error(np.expm1(y_test), np.expm1(pred_log))
print(f'RMSE (log space): {np.square(rmse_log)}')
print(f'RMSE (price)    : {np.square(rmse_price)}')


RMSE (log space): 0.027419774752859026
RMSE (price)    : 223656406011.14636


In [42]:
# Minimal reproducible preprocessing (keeps it simple and robust)
df = test_df.copy()
# drop useless columns if present
for c in ['ID','society','availability']:
    if c in df.columns:
        df.drop(columns=c, inplace=True)

# clean and convert columns
if 'size' in df.columns:
    df['size'] = df['size'].apply(get_size_value)
if 'total_sqft' in df.columns:
    df['total_sqft'] = df['total_sqft'].apply(get_sq_ft)
if 'location' in df.columns:
    df['location'] = df['location'].apply(loc_clean)

# basic feature engineering
# if 'total_sqft' in df.columns and 'price' in df.columns:
#     df['price_per_sqft'] = df.apply(lambda r: (r['price'] / r['total_sqft']) if pd.notna(r['total_sqft']) and r['total_sqft']>0 else np.nan, axis=1)

# drop rows with missing target or essential numeric fields
# df = df.dropna(subset=['price'])
print('After cleaning, rows:', df.shape[0])
df.head()

After cleaning, rows: 2664


,area_type,location,size,total_sqft,bath,balcony
0,Super built-up Area,chamrajpet,2.0,650.0,1.0,1.0
1,Super built-up Area,7th phase jp nagar,3.0,1370.0,2.0,1.0
2,Super built-up Area,whitefield,3.0,1725.0,3.0,2.0
3,Built-up Area,jalahalli,2.0,1000.0,2.0,0.0
4,Plot Area,tc palaya,1.0,1350.0,1.0,0.0


In [44]:
pipe.predict(df)

array([4.44308951, 4.28721732, 4.5283485 , ..., 3.8728388 , 5.23341575,
       3.34238682])

In [45]:
tst_ans = pd.read_csv('Datasets/bengaluru_house_prices.csv')

In [46]:
prc = np.log1p(tst_ans.iloc[10656:]['price'])

In [47]:
np.square(mean_squared_error(prc, pipe.predict(df)))

0.02643908148889753

Next steps (suggestions):
- Try LightGBM / XGBoost for better performance.
- Use target encoding for `location` to reduce dimensionality.
- Add cross-validation and more careful hyperparameter tuning.
- Inspect feature importances and create interaction features where useful.